In [2]:
import pandas as pd
import requests
url = "https://api.themoviedb.org/3/discover/movie?api_key=8265bd1679663a7ea12ac168da84d2e8&include_adult=false&include_video=false&language=en-US&page=1&sort_by=popularity.desc"

In [3]:
df = pd.DataFrame()
var = []


In [4]:
for i in range(1,429):
    url = "https://api.themoviedb.org/3/discover/movie?api_key=8265bd1679663a7ea12ac168da84d2e8&include_adult=false&include_video=false&language=en-US&page={}&sort_by=popularity.desc".format(i)
    response = requests.get(url)
    temp_df = pd.DataFrame(response.json()['results'])[['id','title','overview','release_date','popularity','vote_average','vote_count']]
    var.append(temp_df)


df= pd.concat(var, ignore_index = True )

In [5]:
df.head()

,id,title,overview,release_date,popularity,vote_average,vote_count
0,969681,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,2026-07-29,1550.8083,7.889,1223
1,1368337,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",2026-07-15,886.9314,7.966,2371
2,634649,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,2021-12-15,565.3864,7.900,22526
3,1081003,Supergirl,When an unexpected and ruthless adversary stri...,2026-06-24,469.1073,6.757,1648
4,1339713,Obsession,"After breaking the mysterious ""One Wish Willow...",2026-05-13,314.1366,8.231,4385


In [6]:
from sentence_transformers import SentenceTransformer


In [8]:
model =   SentenceTransformer('bert-base-nli-mean-tokens')
model

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
)

In [22]:
df.dropna(subset = ['overview','title'], axis = 0).reset_index(drop = True)

,id,title,overview,release_date,popularity,vote_average,vote_count
0,969681,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,2026-07-29,1550.8083,7.889,1223
1,1368337,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",2026-07-15,886.9314,7.966,2371
2,634649,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,2021-12-15,565.3864,7.900,22526
3,1081003,Supergirl,When an unexpected and ruthless adversary stri...,2026-06-24,469.1073,6.757,1648
4,1339713,Obsession,"After breaking the mysterious ""One Wish Willow...",2026-05-13,314.1366,8.231,4385
...,...,...,...,...,...,...,...
8555,404403,Jai Jagannath,"When Bhagwan Shri Jagannath and his brother, B...",2007-07-13,4.0931,10.000,1
8556,1664247,Mystic Park,A Porsche driver is stranded in a small countr...,,4.2422,0.000,0
8557,724109,Hole-in-law,An omnibus movie about the jealousy of a man w...,2020-07-06,5.1018,5.786,14
8558,263738,Gulebagavali,A king has two wives. He banishes his first wi...,1955-07-29,4.7789,6.000,1


In [23]:
df  = df[df['overview'].str.strip().str.len()>20]

In [24]:
df = df.reset_index(drop=True)

In [25]:
from sentence_transformers import SentenceTransformer

In [26]:
model=  SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['overview'].tolist(),
                          show_progress_bar = True,
                          batch_size = 64,
                          convert_to_numpy=True
                          )

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/128 [00:00<?, ?it/s]

In [28]:
df['release_date'] = pd.to_datetime(df['release_date'], errors = 'coerce')
df['release_year']  = df['release_date'].dt.year
df.head()

,id,title,overview,release_date,popularity,vote_average,vote_count,release_year
0,969681,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,2026-07-29,1550.8083,7.889,1223,2026.0
1,1368337,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",2026-07-15,886.9314,7.966,2371,2026.0
2,634649,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,2021-12-15,565.3864,7.900,22526,2021.0
3,1081003,Supergirl,When an unexpected and ruthless adversary stri...,2026-06-24,469.1073,6.757,1648,2026.0
4,1339713,Obsession,"After breaking the mysterious ""One Wish Willow...",2026-05-13,314.1366,8.231,4385,2026.0


In [29]:
import numpy as np
df.to_csv("cleaned_movies_data_from_api.csv")
np.save("movie_embeddings.npy", embeddings)

